# 02 – Ablation study (five-fold cold-drug protocol)
**Reproduces:** Section 5.4 and Table 9 (ablation), the parameter counts of the variants
**Responds to:** Reviewer 1 Comment 3; Reviewer 2 Comment 5; Reviewer 3 Comment 7 (items i–v)

All variants use the **same** five cold-drug partitions, label standardization, optimizer, schedule
(Adam, at most 200 epochs, early stopping after 40 epochs without improvement, checkpoint = best validation PCC)
and metrics as the full model in `01_main_cold_drug.ipynb`. The row "Full DualGATSynergy" is read from that notebook's
results (`results/kfold_results_summary.csv`); it is not retrained here.

| Variant (key in the results file) | What changes | Reviewer 3, Comment 7 |
|---|---|---|
| `no_weight_sharing` | two independent GATv2 encoders | (iv) |
| `no_cross_attention_IMCA` | IMCA removed; independently pooled embeddings + context streams are concatenated | (i), (iii) |
| `unidirectional_cross_attn` | IMCA in one direction only (Drug 1 → Drug 2) | (v) |
| `structure_only_no_cb_cell` | ChemBERTa and cell-line streams removed; GATv2 + IMCA kept | (ii) |

Item (vi), alternative pooling operators, was **not** evaluated (stated in the manuscript).

**How to run:** `Runtime → Run all`. Choose what the notebook does with `MODE` (Cell 2):

| `MODE` | What it does | Needs |
|---|---|---|
| `"show_reported"` | prints Table 9 from our saved results and stops (seconds) | only `results/ablation_results.csv` (and `results/kfold_results_summary.csv` for the full-model row) |
| `"evaluate_saved"` (default) | loads our 20 trained ablation models from `checkpoints/ablation/` and re-evaluates them on the test sets; no training, a CPU runtime is enough | data, checkpoints |
| `"train_and_evaluate"` | retrains the variants from scratch (GPU recommended) | data |

> **Note for reviewers who retrain.** Our reported numbers come from one training run per fold. Retraining (`MODE = "train_and_evaluate"`) uses the same
> code, partitions and settings, but GPU operations are not fully deterministic and the library versions may differ, so the numbers you obtain can
> deviate somewhat from those in the manuscript; this is expected. The fold-to-fold SD of the PCC is about 0.07, and the full model, *no weight sharing*,
> *no IMCA* and *unidirectional IMCA* lie within 0.013 PCC of each other, so the order of these four may change between runs. The much lower PCC of the
> *structure-only* variant is large relative to that SD. To verify our numbers exactly, use `MODE = "evaluate_saved"`, which re-evaluates our trained models.

Training is **resumable**: after every (variant, fold) run one result row is written to Drive, and finished runs are skipped when the notebook is
run again, so a disconnected session loses at most the run in progress. Retrained models are written to `checkpoints_retrained/` and never overwrite ours;
retraining gives numbers close to, but not identical with, the reported ones (single run per fold, non-deterministic GPU operations; the reported runs were
made in several Colab sessions).

**Reading the output:** lines ending in "(paper: ...)" show the value reported in the manuscript next to the value computed here.
Standard deviations over folds use n−1.

In [ ]:
# ===== Cell 1: Environment =====
# Colab only. The package versions used for the reported results are pinned in requirements.txt.
!pip install -q rdkit matplotlib pubchempy torch torchvision torchaudio torch-geometric transformers xgboost

# Standard library
import json, os, sys
from pathlib import Path

# Numerics and data
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, ttest_rel

# Deep learning
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import Dataset

# Graph learning
from torch_geometric.data import Data, Batch
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATv2Conv
from torch_geometric.nn.aggr import AttentionalAggregation
from torch_geometric.utils import to_dense_batch

# Chemistry
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem

# Pretrained chemical language model (ChemBERTa)
from transformers import AutoTokenizer, AutoModel

# Classical ML and metrics
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import average_precision_score, mean_squared_error, roc_auc_score
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

RDLogger.DisableLog("rdApp.*")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.1/38.1 MB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 48.9 MB/s eta 0:00:00


In [ ]:
# ===== Cell 2: Configuration =====
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not os.path.ismount("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

# Project folder (see README): on Colab, upload the DualGATSynergy folder to your Drive or change this path.
ROOT = Path("/content/drive/MyDrive/DualGATSynergy") if IN_COLAB else Path(".")
DATA_DIR = ROOT / "data"

# --- Run mode (used in Cells 4-14) ---
MODE = "evaluate_saved"          # "show_reported": print Table 9 from our saved results only (no data, no GPU, seconds)
                                 # "evaluate_saved": load our trained ablation models (no training; CPU is fine)
                                 # "train_and_evaluate": retrain from scratch (GPU recommended; resumable)
assert MODE in ("show_reported", "evaluate_saved", "train_and_evaluate"), f"unknown MODE: {MODE}"
RUN_HEAVY = MODE != "show_reported"   # Cells 4-14 run only when the data are needed
VARIANTS_TO_RUN = ["no_weight_sharing", "no_cross_attention_IMCA",
                   "unidirectional_cross_attn", "structure_only_no_cb_cell"]
FOLDS_TO_RUN = [1, 2, 3, 4, 5]   # when retraining you may run a subset, e.g. one variant and one fold at a time

CKPT_SHIPPED    = ROOT / "checkpoints" / "ablation"             # our trained models, ablation_<variant>_fold<k>.pth (never overwritten)
CKPT_RETRAINED  = ROOT / "checkpoints_retrained" / "ablation"   # written only when MODE = "train_and_evaluate"
RESULTS_SHIPPED = ROOT / "results" / "ablation_results.csv"     # the 20 reported runs (variant x fold)
MAIN_RESULTS    = ROOT / "results" / "kfold_results_summary.csv"  # full model, from notebook 01
# One small CSV per finished (variant, fold) run: this is what makes re-runs resumable.
ROWS_DIR = ROOT / ("results_reproduced" if MODE == "evaluate_saved" else "results_retrained") / "ablation_rows"

SEED = 42
def set_seed(seed=SEED):
    """Called before every (variant, fold) run, so a run does not depend on what was executed before it."""
    torch.manual_seed(seed)   # also seeds CUDA
    np.random.seed(seed)
set_seed()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Mode: {MODE} | device: {DEVICE} | project folder: {ROOT}")

Mounted at /content/drive
Mode: evaluate_saved | device: cpu | project folder: /content/drive/MyDrive/DualGATSynergy


## Table 9 from the saved results (no data or GPU needed)

In [ ]:
# ===== Cell 3: Ablation variants and Table 9 =====
# Flags of the ablation model (Cell 11). Keys are the names used in the results file.
VARIANTS = {
    "no_weight_sharing":         dict(weight_share=False, use_cross_attn=True,  bidirectional=True,  use_chemberta=True,  use_cell_line=True),
    "no_cross_attention_IMCA":   dict(weight_share=True,  use_cross_attn=False, bidirectional=False, use_chemberta=True,  use_cell_line=True),
    "unidirectional_cross_attn": dict(weight_share=True,  use_cross_attn=True,  bidirectional=False, use_chemberta=True,  use_cell_line=True),
    "structure_only_no_cb_cell": dict(weight_share=True,  use_cross_attn=True,  bidirectional=True,  use_chemberta=False, use_cell_line=False),
}
LABELS = {"full": "Full DualGATSynergy",
          "no_weight_sharing": "No weight sharing",
          "no_cross_attention_IMCA": "No IMCA (concatenation)",
          "unidirectional_cross_attn": "Unidirectional IMCA",
          "structure_only_no_cb_cell": "Structure-only"}
PAPER_TABLE6 = {   # (mean PCC, SD of PCC over folds, mean RMSE, mean AUC-ROC) as reported in Table 9
    "full":                      (0.3458, 0.0731, 0.9447, 0.6559),
    "no_weight_sharing":         (0.3332, 0.0708, 0.9520, 0.6562),
    "no_cross_attention_IMCA":   (0.3437, 0.0863, 0.9481, 0.6427),
    "unidirectional_cross_attn": (0.3473, 0.0709, 0.9445, 0.6495),
    "structure_only_no_cb_cell": (0.2053, 0.0350, 0.9839, 0.5842),
}

def summarize_ablation(abl, full=None):
    """Table 9 (mean over folds) plus the paired fold-wise PCC difference to the full model."""
    groups = ([("full", full)] if full is not None else []) + \
             [(key, abl[abl["config"] == key].sort_values("fold")) for key in VARIANTS]
    print(f"{'variant':26s} {'folds':>5s} {'PCC':>7s} {'SD':>7s} {'RMSE':>7s} {'AUC':>7s}")
    for key, d in groups:
        if len(d) == 0:
            continue
        pcc, sd, rmse, auc = d["test_pcc"].mean(), d["test_pcc"].std(), d["test_rmse"].mean(), d["auc"].mean()
        p = PAPER_TABLE6[key]
        print(f"{LABELS[key]:26s} {len(d):5d} {pcc:7.4f} {sd:7.4f} {rmse:7.4f} {auc:7.4f}"
              f"   (paper: {p[0]:.4f} {p[1]:.4f} {p[2]:.4f} {p[3]:.4f})")
    if full is None:
        return
    print("\nPaired difference in test PCC to the full model, per fold (variant - full):")
    full_by_fold = full.set_index("fold")["test_pcc"]
    full_sd = full["test_pcc"].std()
    for key in VARIANTS:
        d = abl[abl["config"] == key].set_index("fold")["test_pcc"]
        folds_common = sorted(set(d.index) & set(full_by_fold.index))
        if len(folds_common) < 2:
            continue
        diff = (d.loc[folds_common] - full_by_fold.loc[folds_common])
        verdict = "smaller" if abs(diff.mean()) < full_sd else "larger"
        print(f"{LABELS[key]:26s} " + " ".join(f"{v:+.3f}" for v in diff.values) +
              f" | mean {diff.mean():+.4f} | better than full in {(diff > 0).sum()}/{len(diff)} folds"
              f" | |mean diff| {verdict} than the fold SD of the full model ({full_sd:.4f})")

if RESULTS_SHIPPED.exists():
    shipped_abl  = pd.read_csv(RESULTS_SHIPPED)
    shipped_full = pd.read_csv(MAIN_RESULTS) if MAIN_RESULTS.exists() else None
    assert len(shipped_abl) == 20 and not shipped_abl.duplicated(["config", "fold"]).any(), "expected 4 variants x 5 folds"
    summarize_ablation(shipped_abl, shipped_full)
else:
    print(f"{RESULTS_SHIPPED} not found: place the reported ablation results there to print Table 9.")
if not RUN_HEAVY:
    print("\nMODE = 'show_reported': nothing else to run. Set MODE = 'evaluate_saved' or 'train_and_evaluate' (Cell 2) to run the models.")

variant                    folds     PCC      SD    RMSE     AUC
Full DualGATSynergy            5  0.3458  0.0731  0.9447  0.6559   (paper: 0.3458 0.0731 0.9447 0.6559)
No weight sharing              5  0.3332  0.0708  0.9520  0.6562   (paper: 0.3332 0.0708 0.9520 0.6562)
No IMCA (concatenation)        5  0.3437  0.0863  0.9481  0.6427   (paper: 0.3437 0.0863 0.9481 0.6427)
Unidirectional IMCA            5  0.3473  0.0709  0.9445  0.6495   (paper: 0.3473 0.0709 0.9445 0.6495)
Structure-only                 5  0.2053  0.0350  0.9839  0.5842   (paper: 0.2053 0.0350 0.9839 0.5842)

Paired difference in test PCC to the full model, per fold (variant - full):
No weight sharing          +0.019 -0.006 -0.016 -0.021 -0.039 | mean -0.0126 | better than full in 1/5 folds | |mean diff| smaller than the fold SD of the full model (0.0731)
No IMCA (concatenation)    +0.017 -0.063 +0.027 -0.065 +0.073 | mean -0.0021 | better than full in 3/5 folds | |mean diff| smaller than the fold SD of the full mod

## Data, molecular representations and cold-drug partition (identical to notebook 01)

In [ ]:
# ===== Cell 4: Load and filter the raw data =====
if RUN_HEAVY:
    # --- 3.1 DrugComb v1.5 ---
    DRUGCOMB_COLS = ["drug_row", "drug_col", "cell_line_name", "synergy_hsa", "study_name", "tissue_name"]
    drugcomb_raw = pd.read_csv(DATA_DIR / "drugcomb_full_v1.5.csv", usecols=DRUGCOMB_COLS, low_memory=False)
    print(f"Raw records:                 {len(drugcomb_raw):,}   (paper: 1,432,351)")

    # Remove non-oncology studies
    NON_ONCOLOGY_KEYWORDS = ["malaria", "covid", "sars", "plasmodium", "tuberculosis"]
    is_non_oncology = (drugcomb_raw["study_name"].astype(str).str.lower()
                       .str.contains("|".join(NON_ONCOLOGY_KEYWORDS), na=False))
    drugcomb_raw = drugcomb_raw[~is_non_oncology].copy()

    # Drop records without an HSA score or a cell-line name (removes nothing in v1.5)
    drugcomb_raw = drugcomb_raw.dropna(subset=["synergy_hsa", "cell_line_name"])
    n_compounds = len(set(drugcomb_raw["drug_row"]) | set(drugcomb_raw["drug_col"]))
    print(f"After filtering:             {len(drugcomb_raw):,}   (paper: 1,432,146) | compounds: {n_compounds:,} (paper: 4,608)")

    # --- 3.2 Compound -> SMILES cache (resolved through PubChem) ---
    cache = pd.read_csv(DATA_DIR / "smiles_cache.csv")        # columns: name, smiles (empty if not resolved)
    drug_to_smiles = dict(zip(cache["name"], cache["smiles"]))
    known_drugs = {name for name, smi in drug_to_smiles.items() if isinstance(smi, str) and smi}
    print(f"Cache: {len(drug_to_smiles)} names, {len(known_drugs)} resolved   (paper: 452 / 418)")

    # Keep only records where BOTH compounds have a SMILES string
    drugcomb = drugcomb_raw[drugcomb_raw["drug_row"].isin(known_drugs) &
                            drugcomb_raw["drug_col"].isin(known_drugs)].copy()
    n_drugs = len(set(drugcomb["drug_row"]) | set(drugcomb["drug_col"]))
    print(f"DrugComb subset: {len(drugcomb):,} records, {n_drugs} drugs   (paper: 61,111 / 276)")

    # --- 3.3 O'Neil ---
    oneil      = pd.read_csv(DATA_DIR / "oneil_summary_idx.csv")
    drugs_meta = pd.read_csv(DATA_DIR / "drugs.csv")
    cells_meta = pd.read_csv(DATA_DIR / "cell_lines.csv")
    drug_map = dict(zip(drugs_meta["id"], drugs_meta["name"]))
    cell_map = dict(zip(cells_meta["id"], cells_meta["name"]))
    oneil["drug_row"]       = oneil["drug_row_idx"].map(drug_map)
    oneil["drug_col"]       = oneil["drug_col_idx"].map(drug_map)
    oneil["cell_line_name"] = oneil["cell_line_idx"].map(cell_map)
    print(f"O'Neil: {len(oneil):,} records   (paper: 22,737) | unmapped names: "
          f"{int(oneil[['drug_row', 'drug_col', 'cell_line_name']].isna().sum().sum())}")

Raw records:                 1,432,351   (paper: 1,432,351)
After filtering:             1,432,146   (paper: 1,432,146) | compounds: 4,608 (paper: 4,608)
Cache: 452 names, 418 resolved   (paper: 452 / 418)
DrugComb subset: 61,111 records, 276 drugs   (paper: 61,111 / 276)
O'Neil: 22,737 records   (paper: 22,737) | unmapped names: 0


In [ ]:
# ===== Cell 5: O'Neil records with SMILES =====
if RUN_HEAVY:
    oneil["smiles_row"] = oneil["drug_row"].map(drug_to_smiles)
    oneil["smiles_col"] = oneil["drug_col"].map(drug_to_smiles)
    oneil_processed = oneil.dropna(subset=["smiles_row", "smiles_col"])[
        ["drug_row", "drug_col", "smiles_row", "smiles_col", "cell_line_name", "synergy_loewe"]].copy()
    oneil_processed.columns = ["drug_1", "drug_2", "smiles_1", "smiles_2", "cell_line", "synergy_score"]
    print(f"O'Neil: {len(oneil_processed):,} of {len(oneil):,} records have SMILES for both drugs   (paper: 22,737 of 22,737)")

O'Neil: 22,737 of 22,737 records have SMILES for both drugs   (paper: 22,737 of 22,737)


In [ ]:
# ===== Cell 6: ChemBERTa embeddings (frozen pretrained model, [CLS] token) =====
if RUN_HEAVY:
    CHEMBERTA_NAME       = "seyonec/ChemBERTa-zinc-base-v1"
    CHEMBERTA_SHIPPED    = DATA_DIR / "drug_to_chemberta.pt"               # 418 compounds x 768 numbers (about 1.3 MB)
    CHEMBERTA_RECOMPUTED = DATA_DIR / "drug_to_chemberta_recomputed.pt"    # never overwrites the shipped file
    RECOMPUTE_CHEMBERTA  = False   # False: load the shipped file (no download, CPU is fine)
                                   # True : recompute with the pretrained model (inference only, no training)

    if not RECOMPUTE_CHEMBERTA:
        drug_to_chemberta = torch.load(CHEMBERTA_SHIPPED, map_location="cpu")
    else:
        tokenizer = AutoTokenizer.from_pretrained(CHEMBERTA_NAME)
        chemberta = AutoModel.from_pretrained(CHEMBERTA_NAME).eval().to(DEVICE)
        print("Model revision:", getattr(chemberta.config, "_commit_hash", "unknown"))

        @torch.no_grad()
        def chemberta_embedding(smiles):
            """[CLS]-token embedding of one SMILES string (truncated to 128 tokens)."""
            inputs = tokenizer(smiles, return_tensors="pt", padding=True, truncation=True, max_length=128).to(DEVICE)
            return chemberta(**inputs).last_hidden_state[:, 0, :].squeeze(0).cpu()

        drug_to_chemberta = {name: chemberta_embedding(drug_to_smiles[name]) for name in sorted(known_drugs)}
        torch.save(drug_to_chemberta, CHEMBERTA_RECOMPUTED)

    CHEMBERTA_DIM = next(iter(drug_to_chemberta.values())).shape[0]
    print(f"ChemBERTa embeddings: {len(drug_to_chemberta)} compounds | dim = {CHEMBERTA_DIM}   (paper pipeline: 418 | 768)")
    assert set(drug_to_chemberta) == known_drugs, "embedding file and SMILES cache do not match"

ChemBERTa embeddings: 418 compounds | dim = 768   (paper pipeline: 418 | 768)


In [ ]:
# ===== Cell 7: Molecular graphs =====
if RUN_HEAVY:
    NODE_FEATURES = ["atomic_number", "degree", "explicit_valence", "implicit_valence", "is_aromatic", "in_ring",
                     "formal_charge", "num_radical_electrons", "hybridization", "chiral_tag", "atomic_mass"]
    NODE_DIM, EDGE_DIM = len(NODE_FEATURES), 1      # edge feature: bond order (1.0, 1.5, 2.0 or 3.0)

    def _valences(atom):
        """(explicit, implicit) valence. GetExplicitValence/GetImplicitValence produced the reported results;
        newer RDKit versions replace them with GetValence(...)."""
        try:
            return atom.GetExplicitValence(), atom.GetImplicitValence()
        except AttributeError:
            return atom.GetValence(Chem.ValenceType.EXPLICIT), atom.GetValence(Chem.ValenceType.IMPLICIT)

    def smiles_to_graph(smiles):
        """SMILES string -> PyG graph (node features: NODE_FEATURES; edge feature: bond order).
        Returns None if the SMILES is missing or cannot be parsed."""
        if not isinstance(smiles, str) or not smiles:
            return None
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None

        features = []
        for atom in mol.GetAtoms():
            explicit_valence, implicit_valence = _valences(atom)
            features.append([
                atom.GetAtomicNum(),
                atom.GetDegree(),
                explicit_valence,
                implicit_valence,
                int(atom.GetIsAromatic()),
                int(atom.IsInRing()),
                atom.GetFormalCharge(),
                atom.GetNumRadicalElectrons(),
                int(atom.GetHybridization()),
                int(atom.GetChiralTag()),       # RDKit chiral tag (none / clockwise / counter-clockwise / other)
                atom.GetMass(),
            ])
        x = torch.tensor(features, dtype=torch.float)

        edge_index, edge_attr = [], []
        for bond in mol.GetBonds():
            s, e = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
            order = bond.GetBondTypeAsDouble()
            edge_index += [(s, e), (e, s)]           # undirected: both directions
            edge_attr  += [order, order]

        if edge_index:
            edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
            edge_attr  = torch.tensor(edge_attr,  dtype=torch.float).view(-1, 1)
        else:                                         # single-atom molecule
            edge_index = torch.empty((2, 0), dtype=torch.long)
            edge_attr  = torch.empty((0, 1),   dtype=torch.float)
        return Data(x=x, edge_index=edge_index, edge_attr=edge_attr)

    # Sanity check on all resolved compounds
    graph_check = {name: smiles_to_graph(drug_to_smiles[name]) for name in known_drugs}
    n_failed   = sum(g is None for g in graph_check.values())
    n_no_bonds = sum(g is not None and g.edge_index.shape[1] == 0 for g in graph_check.values())
    print(f"{len(graph_check)} compounds | failed to parse: {n_failed} | single-atom: {n_no_bonds}   (paper: 0 and 0)")
    assert all(g.x.shape[1] == NODE_DIM for g in graph_check.values() if g is not None)
    del graph_check

418 compounds | failed to parse: 0 | single-atom: 2   (paper: 0 and 0)


In [ ]:
# ===== Cell 8: Combine DrugComb (HSA) and O'Neil (Loewe) =====
if RUN_HEAVY:
    drugcomb_unified = (drugcomb.dropna(subset=["synergy_hsa", "cell_line_name"])
                        [["drug_row", "drug_col", "synergy_hsa", "cell_line_name"]]
                        .rename(columns={"drug_row": "drug_1", "drug_col": "drug_2",
                                         "synergy_hsa": "raw_target", "cell_line_name": "cell_line"}))
    drugcomb_unified["dataset_type"] = "drugcomb"

    oneil_unified = (oneil_processed.dropna(subset=["synergy_score"])
                     [["drug_1", "drug_2", "synergy_score", "cell_line"]]
                     .rename(columns={"synergy_score": "raw_target"}))
    oneil_unified["dataset_type"] = "oneil"

    df_combined = pd.concat([drugcomb_unified, oneil_unified], ignore_index=True)

    # One integer per cell line (order of first appearance) and the list of all compounds
    all_cell_lines   = df_combined["cell_line"].unique()
    cell_line_to_idx = {cl: i for i, cl in enumerate(all_cell_lines)}
    n_cell_lines     = len(all_cell_lines)
    all_drugs        = pd.concat([df_combined["drug_1"], df_combined["drug_2"]]).unique()

    print(f"Records: {len(df_combined):,} (paper: 83,848) | cell-line identifiers: {n_cell_lines} (paper: 251) "
          f"| compounds: {len(all_drugs)} (paper: 309)")

Records: 83,848 (paper: 83,848) | cell-line identifiers: 251 (paper: 251) | compounds: 309 (paper: 309)


In [ ]:
# ===== Cell 9: Five-fold cold-drug partition =====
if RUN_HEAVY:
    COHORTS = ["drugcomb", "oneil"]

    def standardize_labels(train_df, val_df, test_df, fold):
        """Cohort-wise z-scoring of the synergy score into column 'label'.
        Mean and SD of each cohort come from the TRAINING rows only and are then applied to that cohort's
        validation and test rows, so nothing from validation or test enters the statistics."""
        scalers = {}
        for cohort in COHORTS:
            mask = train_df["dataset_type"] == cohort
            if mask.sum() == 0:
                print(f"  [warn] fold={fold}: no '{cohort}' samples in train -> scaler skipped")
                scalers[cohort] = None
                continue
            scalers[cohort] = StandardScaler()
            train_df.loc[mask, "label"] = scalers[cohort].fit_transform(train_df.loc[mask, ["raw_target"]])
        for split_df, split_name in ((val_df, "val"), (test_df, "test")):
            for cohort in COHORTS:
                mask = split_df["dataset_type"] == cohort
                if mask.sum() == 0:
                    continue
                if scalers[cohort] is not None:
                    split_df.loc[mask, "label"] = scalers[cohort].transform(split_df.loc[mask, ["raw_target"]])
                else:   # no training statistics for this cohort: its rows cannot be standardized, so they are dropped
                    print(f"  [warn] fold={fold}: dropping {mask.sum()} '{cohort}' rows from {split_name} (no train scaler)")
                    split_df.drop(split_df[mask].index, inplace=True)
        return scalers


    def build_cold_drug_kfold_partition(df_combined, all_drugs, n_folds=5, val_frac=0.15, seed=42):
        """Drugs (not records) are partitioned. All drugs are shuffled once and cut into n_folds disjoint groups;
        each group is the test set of exactly one fold. Within a fold:
          test  = every record containing at least one test drug
          val   = records with a validation drug and no test drug (46 validation drugs, sampled from the non-test drugs)
          train = all other records (no test drug, no validation drug)
        """
        rng = np.random.RandomState(seed)
        shuffled_drugs = rng.permutation(all_drugs)
        test_groups = np.array_split(shuffled_drugs, n_folds)

        folds = []
        for fold_idx, test_drugs_arr in enumerate(test_groups):
            test_drugs = set(test_drugs_arr)
            remaining_drugs = np.array([d for d in all_drugs if d not in test_drugs])
            fold_rng = np.random.RandomState(seed * 1000 + fold_idx)
            n_val = int(val_frac * len(all_drugs))
            val_drugs = set(fold_rng.choice(remaining_drugs, size=min(n_val, len(remaining_drugs)), replace=False))

            is_test_pair  = df_combined["drug_1"].isin(test_drugs) | df_combined["drug_2"].isin(test_drugs)
            is_val_pair   = (~is_test_pair) & (df_combined["drug_1"].isin(val_drugs) | df_combined["drug_2"].isin(val_drugs))
            is_train_pair = ~is_test_pair & ~is_val_pair
            train_df, val_df, test_df = (df_combined[m].copy() for m in (is_train_pair, is_val_pair, is_test_pair))
            scalers = standardize_labels(train_df, val_df, test_df, fold_idx + 1)

            folds.append({"fold": fold_idx + 1, "train_df": train_df, "val_df": val_df, "test_df": test_df,
                          "test_drugs": test_drugs, "val_drugs": val_drugs,
                          "n_test_drugs": len(test_drugs), "n_val_drugs": len(val_drugs), "scalers": scalers})
        return folds


    def verify_partition(folds, all_drugs):
        """Every drug must be in exactly one test set (no omissions, no duplicates)."""
        all_test_drugs = [d for f in folds for d in f["test_drugs"]]
        counts = pd.Series(all_test_drugs).value_counts()
        n_never, n_duplicated = len(all_drugs) - len(counts), int((counts > 1).sum())
        print(f"Test coverage: {len(counts)}/{len(all_drugs)} drugs | omitted: {n_never} | duplicated: {n_duplicated}")
        assert n_never == 0 and n_duplicated == 0, "not a true partition"
        print("✓ partition verified: every drug is in exactly one test set.")


    def check_or_save_json(path, obj, what):
        """First run: save. Later runs (and reviewers): the rebuilt object must equal the shipped file."""
        path.parent.mkdir(parents=True, exist_ok=True)
        if path.exists():
            assert json.loads(path.read_text(encoding="utf-8")) == obj, f"{what} differs from the shipped file {path.name}"
            print(f"✓ {what} identical to the shipped file")
        else:
            path.write_text(json.dumps(obj, indent=1, ensure_ascii=False), encoding="utf-8")
            print(f"Saved {what} -> {path}")


    folds = build_cold_drug_kfold_partition(df_combined, all_drugs, n_folds=5, seed=42)
    verify_partition(folds, all_drugs)
    check_or_save_json(ROOT / "splits" / "cold_drug_5fold.json",
                       {str(f["fold"]): {"test_drugs": sorted(map(str, f["test_drugs"])),
                                         "val_drugs":  sorted(map(str, f["val_drugs"]))} for f in folds}, "cold-drug split")
    check_or_save_json(ROOT / "splits" / "cell_line_to_idx.json",
                       {str(k): int(v) for k, v in cell_line_to_idx.items()}, "cell-line index")

    print("\nRecords per partition (compare with Table 2):")
    print(pd.DataFrame([{"fold": f["fold"], "train": len(f["train_df"]), "validation": len(f["val_df"]),
                         "test": len(f["test_df"])} for f in folds]).to_string(index=False))

Test coverage: 309/309 drugs | omitted: 0 | duplicated: 0
✓ partition verified: every drug is in exactly one test set.
✓ cold-drug split identical to the shipped file
✓ cell-line index identical to the shipped file

Records per partition (compare with Table 2):
 fold  train  validation  test
    1  31943       16459 35446
    2  44109       12321 27418
    3  31761       20452 31635
    4  37365       16643 29840
    5  33848       22116 27884


In [ ]:
# ===== Cell 10: Dataset =====
if RUN_HEAVY:
    _mol_graph_cache = {}

    def get_cached_graph(smiles):
        """Build each molecular graph once; return a copy so per-sample attributes never leak into the cache."""
        if smiles not in _mol_graph_cache:
            _mol_graph_cache[smiles] = smiles_to_graph(smiles)
        cached = _mol_graph_cache[smiles]
        return cached.clone() if cached is not None else None


    class UnifiedSynergyDataset(Dataset):
        """Sample = (graph of drug 1, graph of drug 2, standardized synergy label).
        A record is kept only if both drugs have a ChemBERTa embedding and a SMILES that RDKit can parse."""
        def __init__(self, dataframe, drug_smiles_dict, drug_chemberta_dict, cell_line_to_idx):
            self.data        = dataframe.reset_index(drop=True)
            self.drug_smiles = drug_smiles_dict
            self.drug_cb     = drug_chemberta_dict
            self.cell_to_idx = cell_line_to_idx

            valid_drugs = {name for name in drug_chemberta_dict
                           if isinstance(drug_smiles_dict.get(name), str)
                           and Chem.MolFromSmiles(drug_smiles_dict[name]) is not None}
            keep = self.data["drug_1"].isin(valid_drugs) & self.data["drug_2"].isin(valid_drugs)
            self.valid_indices = list(self.data.index[keep])
            print(f"Valid samples: {len(self.valid_indices):,} of {len(self.data):,}")

        def __len__(self):
            return len(self.valid_indices)

        def __getitem__(self, idx):
            row = self.data.iloc[self.valid_indices[idx]]
            g1 = get_cached_graph(self.drug_smiles[row["drug_1"]])
            g2 = get_cached_graph(self.drug_smiles[row["drug_2"]])
            g1.cb = self.drug_cb[row["drug_1"]].unsqueeze(0)
            g2.cb = self.drug_cb[row["drug_2"]].unsqueeze(0)
            cl_idx = self.cell_to_idx[row["cell_line"]]
            g1.cell_idx = torch.tensor([cl_idx], dtype=torch.long)
            g2.cell_idx = torch.tensor([cl_idx], dtype=torch.long)
            target = torch.tensor(row["label"], dtype=torch.float)
            return g1, g2, target

## Configurable model and training code

In [ ]:
# ===== Cell 11: DualGATSynergy with switchable components =====
if RUN_HEAVY:
    class DualGATSynergyAblation(nn.Module):
        """DualGATSynergy (Cell 10 of notebook 01) with switchable components. With the default flags it is exactly the
        full model: same modules, same names, same parameter count, so the full model's checkpoints load into it.
          weight_share   : True  = one GATv2 encoder for both drugs; False = two independent encoders
          use_cross_attn : True  = IMCA; False = IMCA removed (late fusion by concatenation)
          bidirectional  : True  = IMCA in both directions; False = Drug 1 -> Drug 2 only
          use_chemberta  : ChemBERTa streams on/off
          use_cell_line  : cell-line stream on/off
        """
        def __init__(self, num_atom_features, chemberta_dim, n_cell_lines, hidden_dim=64, heads=2,
                     weight_share=True, use_cross_attn=True, bidirectional=True,
                     use_chemberta=True, use_cell_line=True):
            super().__init__()
            self.weight_share   = weight_share
            self.use_cross_attn = use_cross_attn
            self.bidirectional  = bidirectional and use_cross_attn
            self.use_chemberta  = use_chemberta
            self.use_cell_line  = use_cell_line

            self.conv1 = GATv2Conv(num_atom_features, hidden_dim, heads=heads, edge_dim=1, dropout=0.2)
            self.conv2 = GATv2Conv(hidden_dim * heads, hidden_dim, heads=1, concat=False, edge_dim=1, dropout=0.2)
            if not weight_share:     # second, independent encoder for Drug 2
                self.conv1_b = GATv2Conv(num_atom_features, hidden_dim, heads=heads, edge_dim=1, dropout=0.2)
                self.conv2_b = GATv2Conv(hidden_dim * heads, hidden_dim, heads=1, concat=False, edge_dim=1, dropout=0.2)

            self.pool = AttentionalAggregation(gate_nn=nn.Sequential(
                nn.Linear(hidden_dim, 32), nn.ReLU(), nn.Linear(32, 1)))

            if use_cross_attn:
                self.cross_attention = nn.MultiheadAttention(embed_dim=hidden_dim, num_heads=4, dropout=0.1, batch_first=True)
            if use_chemberta:
                self.chemberta_proj = nn.Sequential(
                    nn.Linear(chemberta_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.ReLU(), nn.Dropout(0.3))
            if use_cell_line:
                self.cell_embedding = nn.Embedding(n_cell_lines, hidden_dim)

            # The two pooled molecular embeddings are always present; the other streams depend on the flags.
            n_streams = 2
            if use_cross_attn:
                n_streams += 2 if self.bidirectional else 1
            if use_chemberta:
                n_streams += 2
            if use_cell_line:
                n_streams += 1
            self.predictor = nn.Sequential(
                nn.Linear(hidden_dim * n_streams, 256), nn.ReLU(), nn.Dropout(0.3),
                nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.3),
                nn.Linear(128, 64), nn.ReLU(),
                nn.Linear(64, 1))

        def encode_atoms(self, x, edge_index, edge_attr, drug=1):
            conv1, conv2 = (self.conv1, self.conv2) if (drug == 1 or self.weight_share) else (self.conv1_b, self.conv2_b)
            x = F.elu(conv1(x, edge_index, edge_attr=edge_attr))
            x = F.dropout(x, p=0.2, training=self.training)
            return conv2(x, edge_index, edge_attr=edge_attr)

        def forward(self, d1, d2):
            atoms1 = self.encode_atoms(d1.x, d1.edge_index, d1.edge_attr, drug=1)
            atoms2 = self.encode_atoms(d2.x, d2.edge_index, d2.edge_attr, drug=2)
            emb1, emb2 = self.pool(atoms1, d1.batch), self.pool(atoms2, d2.batch)
            streams, w_1to2, w_2to1 = [emb1, emb2], None, None

            if self.use_cross_attn:
                dense1, mask1 = to_dense_batch(atoms1, d1.batch)
                dense2, mask2 = to_dense_batch(atoms2, d2.batch)
                attn_1to2, w_1to2 = self.cross_attention(dense1, dense2, dense2, key_padding_mask=~mask2)
                streams.append(self.pool(attn_1to2[mask1], d1.batch))
                if self.bidirectional:
                    attn_2to1, w_2to1 = self.cross_attention(dense2, dense1, dense1, key_padding_mask=~mask1)
                    streams.append(self.pool(attn_2to1[mask2], d2.batch))
            if self.use_chemberta:
                streams += [self.chemberta_proj(d1.cb), self.chemberta_proj(d2.cb)]
            if self.use_cell_line:
                streams.append(self.cell_embedding(d1.cell_idx))
            return self.predictor(torch.cat(streams, dim=1)), (w_1to2, w_2to1)


    MODEL_CFG = dict(num_atom_features=NODE_DIM, chemberta_dim=CHEMBERTA_DIM, n_cell_lines=n_cell_lines, hidden_dim=64)

    def count_parameters(**flags):
        m = DualGATSynergyAblation(**MODEL_CFG, **flags)
        return sum(p.numel() for p in m.parameters() if p.requires_grad)

    n_full = count_parameters()
    print(f"{'Full DualGATSynergy':26s} {n_full:>9,} trainable parameters   (paper: 260,482)")
    assert n_full == 260_482, "default flags must give exactly the full model"
    for key, flags in VARIANTS.items():
        print(f"{LABELS[key]:26s} {count_parameters(**flags):>9,} trainable parameters")

Full DualGATSynergy          260,482 trainable parameters   (paper: 260,482)
No weight sharing            280,642 trainable parameters
No IMCA (concatenation)      211,074 trainable parameters
Unidirectional IMCA          244,098 trainable parameters
Structure-only               145,922 trainable parameters


In [ ]:
# ===== Cell 12: Training and evaluation functions =====
if RUN_HEAVY:
    def evaluate_model(model, loader):
        """RMSE, PCC and the raw predictions / targets of `model` on `loader`."""
        model.eval()
        all_preds, all_targets = [], []
        with torch.no_grad():
            for batch in loader:
                d1, d2, target = batch[0].to(DEVICE), batch[1].to(DEVICE), batch[2].to(DEVICE)
                pred, _ = model(d1, d2)
                all_preds.extend(pred.cpu().numpy().flatten())
                all_targets.extend(target.cpu().numpy().flatten())
        rmse = np.sqrt(mean_squared_error(all_targets, all_preds))
        pcc, _ = pearsonr(all_targets, all_preds)
        return float(rmse), float(pcc), all_preds, all_targets


    def train_one_fold(model, train_loader, val_loader, checkpoint_path, num_epochs=200, patience_epochs=40):
        """Adam + Smooth-L1. ONLY the validation set drives the LR scheduler, early stopping and checkpoint selection
        (best validation PCC); the test set is never used here. Returns the best validation PCC."""
        optimizer = torch.optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-4)
        scheduler = ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)
        criterion = nn.SmoothL1Loss()
        best_val_pcc, epochs_since_improve = -1.0, 0

        for epoch in range(num_epochs):
            model.train()
            total_loss = 0
            for batch in train_loader:
                d1, d2, targets = batch[0].to(DEVICE), batch[1].to(DEVICE), batch[2].to(DEVICE)
                optimizer.zero_grad()
                predictions, _ = model(d1, d2)
                loss = criterion(predictions, targets.view(-1, 1))
                loss.backward()
                optimizer.step()
                total_loss += loss.item()
            avg_loss = total_loss / len(train_loader)

            if (epoch + 1) % 2 == 0:                          # validate every 2 epochs
                val_rmse, val_pcc, _, _ = evaluate_model(model, val_loader)
                scheduler.step(val_pcc)
                print(f"  Epoch {epoch+1:03d} | Loss: {avg_loss:.4f} | Val RMSE: {val_rmse:.4f} | Val PCC: {val_pcc:.4f}")
                if val_pcc > best_val_pcc:
                    best_val_pcc, epochs_since_improve = val_pcc, 0
                    torch.save(model.state_dict(), checkpoint_path)
                else:
                    epochs_since_improve += 2
                if epochs_since_improve >= patience_epochs:
                    print(f"  Early stopping at epoch {epoch+1}")
                    break
        return best_val_pcc

## Run the ablation (resumable)

In [ ]:
# ===== Cell 13: Run the variants – load our models (or retrain); the test set is used exactly once per run =====
if RUN_HEAVY:
    TABLE2_SIZES = {   # fold: (train, validation, test, test drugs) records; must equal Table 2
        1: (31943, 16459, 35446, 62), 2: (44109, 12321, 27418, 62), 3: (31761, 20452, 31635, 62),
        4: (37365, 16643, 29840, 62), 5: (33848, 22116, 27884, 61)}
    for f in folds:
        got = (len(f["train_df"]), len(f["val_df"]), len(f["test_df"]), f["n_test_drugs"])
        assert got == TABLE2_SIZES[f["fold"]], f"fold {f['fold']}: {got} != {TABLE2_SIZES[f['fold']]}"
    print("✓ partitions match Table 2 (the same partitions as the full model)")
    if MODE == "train_and_evaluate":
        print("Note: retrained numbers may deviate somewhat from the reported ones (single run per fold, non-deterministic GPU operations,\n"
              "library versions). The fold-to-fold SD of the PCC is about 0.07; see the note at the top of this notebook.")

    def make_dataset(df):
        return UnifiedSynergyDataset(df, drug_to_smiles, drug_to_chemberta, cell_line_to_idx)

    ROW_COLS = ["config", "fold", "val_pcc", "test_rmse", "test_pcc", "auc", "aupr"]
    ROWS_DIR.mkdir(parents=True, exist_ok=True)
    row_path  = lambda variant, k: ROWS_DIR / f"{variant}__fold{k}.csv"
    ckpt_path = lambda directory, variant, k: directory / f"ablation_{variant}_fold{k}.pth"
    reported  = pd.read_csv(RESULTS_SHIPPED) if RESULTS_SHIPPED.exists() else None

    skipped = []
    for variant in VARIANTS_TO_RUN:
        for f in folds:
            k = f["fold"]
            if k not in FOLDS_TO_RUN:
                continue
            if row_path(variant, k).exists():
                print(f"⏭  {variant} | fold {k}: already done")
                continue
            print(f"\n{'='*60}\n{variant} | fold {k}/5\n{'='*60}")

            set_seed(SEED)
            model = DualGATSynergyAblation(**MODEL_CFG, **VARIANTS[variant]).to(DEVICE)
            test_ds     = make_dataset(f["test_df"])
            test_loader = DataLoader(test_ds, batch_size=128, shuffle=False)

            if MODE == "train_and_evaluate":
                train_loader = DataLoader(make_dataset(f["train_df"]), batch_size=128, shuffle=True)
                val_loader   = DataLoader(make_dataset(f["val_df"]),   batch_size=128, shuffle=False)
                ckpt = ckpt_path(CKPT_RETRAINED, variant, k)
                ckpt.parent.mkdir(parents=True, exist_ok=True)
                best_val_pcc = train_one_fold(model, train_loader, val_loader, ckpt)
            else:
                ckpt = ckpt_path(CKPT_SHIPPED, variant, k)
                if not ckpt.exists():
                    print(f"  [skip] checkpoint not found: {ckpt}")
                    skipped.append((variant, k))
                    continue
                best_val_pcc = float("nan")

            model.load_state_dict(torch.load(ckpt, map_location=DEVICE))        # best-validation checkpoint
            test_rmse, test_pcc, test_preds, test_targets = evaluate_model(model, test_loader)   # the only use of the test set

            kept = f["test_df"].iloc[test_ds.valid_indices]
            assert np.allclose(kept["label"].values, test_targets, atol=1e-5), "rows and labels are misaligned"
            labels_cls = (kept["raw_target"].values > 10).astype(int)            # positive: raw score > 10 (cohort's native scale)
            auc, aupr = roc_auc_score(labels_cls, test_preds), average_precision_score(labels_cls, test_preds)

            row = {"config": variant, "fold": k, "val_pcc": best_val_pcc, "test_rmse": test_rmse,
                   "test_pcc": test_pcc, "auc": auc, "aupr": aupr}
            tmp = row_path(variant, k).with_suffix(".tmp")                        # write, then rename: no half-written rows
            pd.DataFrame([row])[ROW_COLS].to_csv(tmp, index=False)
            os.replace(tmp, row_path(variant, k))
            print(f"  ✓ saved | {variant} fold {k} -> Test PCC: {test_pcc:.4f} | RMSE: {test_rmse:.4f} | AUC: {auc:.4f}")

            del model
            if DEVICE == "cuda":
                torch.cuda.empty_cache()

    # Compare EVERY finished run with the reported results, including runs finished in an earlier session
    if MODE == "evaluate_saved" and reported is not None:
        done = pd.concat([pd.read_csv(p) for p in sorted(ROWS_DIR.glob("*__fold*.csv"))], ignore_index=True)
        cmp = done.merge(reported, on=["config", "fold"], suffixes=("", "_reported"))
        metrics = ["test_pcc", "test_rmse", "auc", "aupr"]
        # AUPR depends on the ranking at the very top of the list, so for a model close to chance (e.g. structure-only, fold 5, AUC 0.499)
        # it moves more under floating-point differences between hardware/library versions than PCC, RMSE or AUC do.
        tol = {"test_pcc": 1e-3, "test_rmse": 1e-3, "auc": 1e-3, "aupr": 5e-3}
        print(f"\nRe-evaluated runs compared with the reported results ({len(cmp)} runs); largest absolute difference per metric:")
        print("  " + " | ".join(f"{c}: {(cmp[c] - cmp[c + '_reported']).abs().max():.1e}" for c in metrics))
        off = [(r["config"], int(r["fold"]), c, round(r[c], 4), round(r[c + "_reported"], 4))
               for _, r in cmp.iterrows() for c in metrics if abs(r[c] - r[c + "_reported"]) >= tol[c]]
        if not off and not skipped:
            print("✓ all re-evaluated models reproduce the reported results (PCC, RMSE, AUC within 1e-3; AUPR within 5e-3)")
        else:
            for config, fold, metric, new, old in off:
                print(f"⚠ {config} fold {fold}: {metric} = {new} here, {old} reported")
            if skipped:
                print(f"⚠ checkpoints missing: {skipped}")

✓ partitions match Table 2 (the same partitions as the full model)
⏭  no_weight_sharing | fold 1: already done
⏭  no_weight_sharing | fold 2: already done
⏭  no_weight_sharing | fold 3: already done
⏭  no_weight_sharing | fold 4: already done
⏭  no_weight_sharing | fold 5: already done
⏭  no_cross_attention_IMCA | fold 1: already done
⏭  no_cross_attention_IMCA | fold 2: already done
⏭  no_cross_attention_IMCA | fold 3: already done
⏭  no_cross_attention_IMCA | fold 4: already done
⏭  no_cross_attention_IMCA | fold 5: already done
⏭  unidirectional_cross_attn | fold 1: already done
⏭  unidirectional_cross_attn | fold 2: already done
⏭  unidirectional_cross_attn | fold 3: already done
⏭  unidirectional_cross_attn | fold 4: already done
⏭  unidirectional_cross_attn | fold 5: already done
⏭  structure_only_no_cb_cell | fold 1: already done
⏭  structure_only_no_cb_cell | fold 2: already done
⏭  structure_only_no_cb_cell | fold 3: already done
⏭  structure_only_no_cb_cell | fold 4: already 

## Summary of this run

In [ ]:
# ===== Cell 14: Table 9 from this run (compare with Cell 3) =====
if RUN_HEAVY:
    paths = sorted(ROWS_DIR.glob("*__fold*.csv"))
    if not paths:
        print(f"No finished runs in {ROWS_DIR} (missing checkpoints?). Table 9 from our saved results: Cell 3.")
    else:
        this_run = pd.concat([pd.read_csv(p) for p in paths], ignore_index=True)
        full = pd.read_csv(MAIN_RESULTS) if MAIN_RESULTS.exists() else None
        print(f"{len(this_run)} of {len(VARIANTS) * 5} runs found in {ROWS_DIR}\n")
        summarize_ablation(this_run, full)

        if MODE == "train_and_evaluate" and RESULTS_SHIPPED.exists():
            ref = pd.read_csv(RESULTS_SHIPPED)
            print("\nMean test PCC, this run vs. reported (small deviations are expected, see the note at the top):")
            for key in VARIANTS:
                a, b = this_run[this_run["config"] == key], ref[ref["config"] == key]
                if len(a):
                    print(f"{LABELS[key]:26s} this run {a['test_pcc'].mean():.4f} | reported {b['test_pcc'].mean():.4f} "
                          f"| difference {a['test_pcc'].mean() - b['test_pcc'].mean():+.4f}   ({len(a)} folds)")

        out_csv = ROWS_DIR.parent / "ablation_results.csv"      # never touches the reported results in results/
        this_run.sort_values(["config", "fold"]).to_csv(out_csv, index=False)
        print(f"\nSaved -> {out_csv}")

20 of 20 runs found in /content/drive/MyDrive/DualGATSynergy/results_reproduced/ablation_rows

variant                    folds     PCC      SD    RMSE     AUC
Full DualGATSynergy            5  0.3458  0.0731  0.9447  0.6559   (paper: 0.3458 0.0731 0.9447 0.6559)
No weight sharing              5  0.3332  0.0708  0.9520  0.6562   (paper: 0.3332 0.0708 0.9520 0.6562)
No IMCA (concatenation)        5  0.3437  0.0863  0.9481  0.6427   (paper: 0.3437 0.0863 0.9481 0.6427)
Unidirectional IMCA            5  0.3473  0.0709  0.9445  0.6495   (paper: 0.3473 0.0709 0.9445 0.6495)
Structure-only                 5  0.2053  0.0350  0.9839  0.5842   (paper: 0.2053 0.0350 0.9839 0.5842)

Paired difference in test PCC to the full model, per fold (variant - full):
No weight sharing          +0.019 -0.006 -0.016 -0.021 -0.039 | mean -0.0126 | better than full in 1/5 folds | |mean diff| smaller than the fold SD of the full model (0.0731)
No IMCA (concatenation)    +0.017 -0.063 +0.027 -0.065 +0.073 | mean